# Run This Top Cell Beforehand; I was unable to install the most recent versions of Selenium and undetected_chromedriver via the requirements.txt package, but this script *will* requier updated versions of them.

In [1]:
!pip install undetected-chromedriver --upgrade
!pip install selenium==4.12.0

  Using cached undetected_chromedriver-3.5.5-py3-none-any.whl
  Using cached trio-0.33.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------- ----------------------------- 2.6/9.7 MB 16.9 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.7 MB 19.1 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.7 MB 19.0 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.7 MB 19.0 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.7 MB 19.0 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.7 MB 19.0 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 7.7 MB/s  0:00:01
Using cached trio-0.33.0-py3-none-any.whl (510 kB)
Using cached trio_websocket-0.12.2-py3-none-any.whl (21 kB)
Using cached 

In [2]:
import undetected_chromedriver as uc

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys

import time
import random
import os
import ast
import sys
import re

import unicodedata


import pycountry
from convenient_pickle import *

from bs4 import BeautifulSoup
import re
import pyperclip
import unicodedata

In [3]:
def remove_diacritics(text):
    normalized = unicodedata.normalize('NFD', text)
    return ''.join(
        c for c in normalized
        if unicodedata.category(c) != 'Mn'
    )

def urlify(text):
    text = remove_diacritics(text)
    text = ''.join([i for i in text.lower() if i in 'abcdefghijklmnopqrstuvwxyz'])
    return text

In [4]:
#placename_tiktok_data = load_pickle(os.getcwd()+'/placename_tiktok_data.pkl')
#capital_tiktok_places = load_pickle(os.getcwd()+'/capitalized_placenames')

In [5]:
def interpret_tiktok_postcount(instr):
    instr = instr.split()[0]
    numstr = ''
    thoustr = ''
    for i in instr: 
        if i in '1234567890.': 
            numstr += i
        else: 
            thoustr += i
            break
    numstr = float(numstr)
    if thoustr.lower() == 'k':
        thoustr = 1000
    elif thoustr.lower() == 'm':
        thoustr = 1000000
    elif thoustr.lower() == 'b':
        thoustr = 1000000000
    else: 
        thoustr = 1
    return int(numstr * thoustr)

In [6]:
def prettify_tiktok_page(video):
    info_list = []
    try:
        #print(video)
        soup = BeautifulSoup(video,'html.parser')

        video_poster_username = soup.find_all('span',attrs={'data-e2e': 'browse-username'})[0].text

        
        description = soup.find_all('div',attrs={'data-e2e': 'browse-video-desc'})[0]
        description_text = '\n'.join([i.text for i in description.find_all('span',attrs={'data-e2e':"new-desc-span"})])

        #testing this out
        current_soup = soup.find('div', attrs={'data-e2e':'search-comment-container'})
        soup_hashtags = ' '.join([i.text for i in current_soup.find_all('a',attrs={'data-e2e': 'search-common-link'})])
        description_text += '\n' + soup_hashtags
        #comment the above out if it doesn't work

        
        like_count_text = soup.find_all('strong',attrs={'data-e2e': 'browse-like-count'})[0].text
        comment_count_text = soup.find_all('strong',attrs={'data-e2e': 'browse-comment-count'})[0].text
        bookmark_count_text = soup.find_all('strong',attrs={'data-e2e': 'undefined-count'})[0].text
        date_text = soup.find_all('span',attrs={'data-e2e': 'browser-nickname'})[0].find_all('span')[-1].text
        link_text = soup.find_all('p',attrs={'data-e2e': 'browse-video-link'})[0].text.split('?')[0]
        placelist = soup.select('p[class*="PAnchorTagName"]')
        if len(placelist)==0: 
            place = None
        else: 
            place = placelist[0].text
        
        comments = soup.select('div[class*="DivCommentContentContainer"]')
        comment_list = []
        for comment in comments: 
            href = comment.find('a',href=True)['href']
            comment_likes = comment.find_all('span', attrs={'data-e2e':'comment-like-count'})[0].text
            comment_text = comment.select('p[data-e2e*="comment-level"]')[0].text
            comment_time = comment.select('span[data-e2e*="comment-time"]')[0].text
            comment_username = comment.select('span[data-e2e*="comment-username"]')[0].text
            comment_list.append({'href':href, "comment_likes":comment_likes, "comment_text":comment_text,"comment_time":comment_time,
                                 "comment_username":comment_username})
        info_list.append((video_poster_username,description_text,like_count_text,comment_count_text,bookmark_count_text,place,date_text,link_text,comment_list))
        #print('made it at least one')
    except:
        pass
    return info_list

def collect_comment_text(inlist):
    outlist = []
    for comment in inlist: 
        outlist.append(comment['comment_text'])
    return ' '.join(outlist)



# If you're using a VPN, make sure to activate it before you run this next cell.

In [7]:
options = uc.ChromeOptions()

brave_path = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"
options.binary_location = brave_path

options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--start-maximized")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-backgrounding-occluded-windows")
options.add_argument("--disable-renderer-backgrounding")
options.add_argument("--TZ=US/Central") #set this to whatever time zone your VPN is in (if using a VPN) or your time zone (if not).
options.add_argument("--incognito")
options.add_argument("--mute-audio")

driver = uc.Chrome(
    options=options,
    headless=False,
    use_subprocess=True,
    #version_main=146
)


wait = WebDriverWait(driver, 20)

# If you only want to scrape a one country or few countries, comment out the first "placename_list" below and uncomment the second or third placename_list.

In [8]:
#placename_tiktok_data.keys()

# placename_list =['Albania', 'Armenia', 'Azerbaijan', 'Bosnia', 'Bulgaria', 
#                  'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'England', 'Estonia', 
#                  'Finland', 'France', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 
#                  'Kosovo', 'Latvia', 'Lithuania', 'Luxembourg', 'Macedonia', 'Malta', 
#                  'Moldova', 'Montenegro', 'Netherlands', 'Norway', 'Poland', 'Portugal', 
#                  'Romania', 'Scotland', 'Serbia', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 
#                  'Turkey', 'Wales']

#If you only want to scrape a single country (or a few countries), 
placename_list = ['Albania']

#placename_list = ['Austria','Belgium','Switzerland','Germany']

print(placename_list)

['Albania']


# You can scrape whatever tag you want by altering the keyword and/or changing the truth value of the "reverse" variable. It will save them under new filenames. 

# Additionally, before running this cell, I would recommend navigating to a tag on TikTok, such as http://tiktok.com/tag/travelbarcelona , then scroll through a couple of the videos using the down arrow key until you suddenly can't. Then hit exit the video (not the browser or the tab; you just want to return to the visitbarcelona tag) and then fill out the prompt that TikTok gives you. Otherwise the prompt will interfere with future scraping. 

# Finally, please note that this process can require some babysitting; sometimes the keyboard inputs will get stuck while scrolling through videos, so you'll have to scroll to the next video by clicking (I do not include an automated version of the clicking as I found that using automated mouse commands on TikTok tends to trigger the website's IP blocking).

In [9]:
#since there are only 150 videos per tag most of the time, 
#having values more than 200 basically gives you a buffer in case the scraping gets stuck.

video_count=1000

scraped_placename_dict = dict()
searchbar = False


reverse=False
keyword = "travel"


for country in list(placename_list):
    actions=ActionChains(driver)
    use_country = country.lower()
    dict_of_country = dict()



    #navigate to search bar
    if searchbar:
        actions.key_down(Keys.ESCAPE).pause(random.uniform(0,0.3)).key_up(Keys.ESCAPE).perform()
        driver.get(driver.current_url)
        time.sleep(random.uniform(5,7))
    
        
        ##search out the country
        active = driver.switch_to.active_element.get_attribute('class')
        while "tuxbutton" not in active.lower(): 
            actions = ActionChains(driver)
            arrowwait = random.uniform(0,0.3)
            actions.key_down(Keys.TAB).pause(arrowwait).key_up(Keys.TAB).perform()
            active = driver.switch_to.active_element.get_attribute('class')
        arrowwait = random.uniform(0,0.3)
        actions.key_down(Keys.SPACE).pause(arrowwait).key_up(Keys.SPACE).perform()
        arrowwait = random.uniform(0,0.3)
        actions.key_down(Keys.CONTROL).pause(arrowwait).send_keys("a").key_up(Keys.CONTROL).perform()
        arrowwait = random.uniform(0,0.3)
        time.sleep(random.uniform(0.5,1))
        actions.key_down(Keys.BACK_SPACE).pause(arrowwait).key_up(Keys.BACK_SPACE).perform()
    
        #type in query and send
        if reverse:
            searchstring = f"{use_country} {keyword} 2025"
        else: 
            searchstring = f"{keyword} {use_country} 2025"
        for char in f"{keyword} {use_country} 2025": 
            actions.send_keys(char)
            actions.pause(random.uniform(0.05,0.25))
        
        actions.send_keys(Keys.ENTER)
        actions.perform()
    else: 
        if not reverse: 
            url = f"https://www.tiktok.com/tag/{keyword}{use_country}"
        else: 
            url = f"https://www.tiktok.com/tag/{use_country}{keyword}"
        driver.get(url)
    #wait for page to load
    time.sleep(random.uniform(5,7))
    check = video_count


    
    #load in the first video
    old_url = driver.current_url
    new_url = driver.current_url
    while old_url == new_url:
        if searchbar:
            look_in = 'data-e2e'
            lookfor = 'search_top_item'
        else: 
            look_in = 'data-e2e'
            lookfor = 'challenge-item'
        active = driver.switch_to.active_element.get_attribute(look_in)
        while active == None or lookfor not in active.lower():
            actions=ActionChains(driver)
            arrowwait = random.uniform(0,0.3)
            actions.key_down(Keys.TAB).pause(arrowwait).key_up(Keys.TAB).perform()
            active = driver.switch_to.active_element.get_attribute(look_in)
            #print(active)
        actions.send_keys(Keys.SPACE).perform()
        new_url = driver.current_url
    
    location_list = []
    
    last_wasnt_relevant = False
    screwup=0
    #go through the checks and 
    for i in range(check):
        time.sleep(random.uniform(1,4))
        test = driver.page_source
        current_url = driver.current_url
        prettified_data = prettify_tiktok_page(test)
        if prettified_data == []:
            continue
        
        
        check_description = remove_diacritics(prettified_data[0][1]).lower()
        check_comments = remove_diacritics(collect_comment_text(prettified_data[0][-1])).lower()
        if prettified_data[0][5] == None:
            check_location = ''
        else:
            check_location = remove_diacritics(prettified_data[0][5]).lower()
    
        #proper_location = placename_travel_tags[country][hashtag][0].lower()
        #hashtag_location = placename_travel_tags[country][hashtag][1].lower()
        country_location = country.lower()
    
                                           
        #location_mentioned = proper_location in check_description or proper_location in check_comments or proper_location in check_location
        #hashtag_mentioned = hashtag_location in check_description or hashtag_location in check_comments or hashtag_location in check_location
        country_mentioned = country_location in check_description or country_location in check_comments or country_location in check_location
    
        relevant = country_mentioned
        #if relevant: 
        
        #    last_wasnt_relevant=False
        #elif last_wasnt_relevant:
        #    break
        #else:
        #    last_wasnt_relevant=True
        actions = ActionChains(driver)
        arrowwait = random.uniform(0.25,0.75)
        actions.key_down(Keys.SPACE).pause(arrowwait).key_up(Keys.SPACE).perform()
        actions.key_down(Keys.ARROW_DOWN).pause(arrowwait).key_up(Keys.ARROW_DOWN).perform()
        try:
            WebDriverWait(driver,10).until(lambda _ : driver.current_url != current_url)
            if relevant:
                location_list.append(prettified_data)
        except: 
            screwup += 1
            print(f"There have been {screwup} screwups in {use_country}")
        if screwup >= 5:
            break
    current_dir = os.getcwd()
    scraped_placename_dict[country] = location_list
    filename = f"{country}_placenames_records_{keyword}"
    if keyword == "travel" and reverse == False: 
        filename = f"{country}_placenames_records"
    if reverse:
        filename += "_reversed"
    if searchbar: 
        filename+="_searchbar"
    dump_pickle(os.getcwd()+'/saved_data_country_names/',filename,location_list,warn=False)
    os.chdir(current_dir)
    print(country)
    print(len(location_list))

There have been 1 screwups in albania
There have been 2 screwups in albania
There have been 3 screwups in albania
There have been 4 screwups in albania
There have been 5 screwups in albania
Albania
167


In [13]:
#This cell will consolidate all of your scraped data from the same country into a single file.


currentdir = os.getcwd()
for country in placename_list: 
    
    # base_albania = load_pickle(os.getcwd()+f'/saved_data_country_names/{country}_placenames_records')
    # explore_albania = load_pickle(os.getcwd()+f'/saved_data_country_names/{country}_placenames_records_explore')
    # visit_albania = load_pickle(os.getcwd()+f'/saved_data_country_names/{country}_placenames_records_visit')
    # tourism_reverse_albania = load_pickle(os.getcwd()+f'/saved_data_country_names/{country}_placenames_records_tourism_reversed')
    # discover_albania = load_pickle(os.getcwd()+f'/saved_data_country_names/{country}_placenames_records_discover')

    records_list = [i for i in os.listdir(os.getcwd()+f'/saved_data_country_names/') if country in i]
    
    filelist = [load_pickle(os.getcwd()+f'/saved_data_country_names/'+i) for i in records_list] 
    
    albania_list = []
    albania_urls = dict()
    
    #for file in [base_albania, explore_albania, visit_albania, tourism_reverse_albania, discover_albania]:
    for file in filelist:
        for video in file: 
            url = video [0][7]
            if url not in albania_urls: 
                albania_list.append(video)
                albania_urls[url] = True
    dump_pickle(os.getcwd(),f'/saved_data_countrynames_consolidated/{country}_countrynames_consolidated.pkl', albania_list)
    print(f"{country}: {len(albania_list)}")

Albania: 167
